# lulc-toolkit — Colab Tutorial

This notebook uses the [`lulc-toolkit`](https://github.com/YOUR_USERNAME/lulc-toolkit) package to download Google Dynamic World land-cover data per district/year, render styled PNG maps, compute area statistics, and generate comparison charts.

Replace `YOUR_USERNAME` in the install cell below with your actual GitHub username once you've pushed the package.

## 1. Install the package

In [ ]:
!pip install -q git+https://github.com/YOUR_USERNAME/lulc-toolkit.git
!apt-get install unrar -y -qq   # needed only if your shapefile archive is a .rar


## 2. Import functions

In [ ]:
from lulc_toolkit import (
    authenticate_ee, mount_drive_readonly, find_archive_in_drive,
    extract_shapefile_archive, load_shapefile,
    download_dynamicworld_for_years, DW_CLASSES, DW_CLASS_NAMES,
    generate_all_maps, build_statistics_table, build_pivot_table,
    generate_all_charts, zip_and_download,
)
import os


## 3. Authenticate Earth Engine and load your shapefile
Set `PROJECT_ID` to your Earth Engine project and `SEARCH_KEYWORD` to a substring matching your shapefile archive's filename in Google Drive.

In [ ]:
PROJECT_ID = 'dyalaali'
SEARCH_KEYWORD = 'diyala'

authenticate_ee(PROJECT_ID)
mount_drive_readonly()

archive_path = find_archive_in_drive(SEARCH_KEYWORD)
if archive_path is None:
    raise FileNotFoundError(f"No archive containing '{SEARCH_KEYWORD}' found in Drive.")
print('✓ Archive found:', archive_path)

shp_path = extract_shapefile_archive(archive_path, '/content/shapefile_data')
gdf = load_shapefile(shp_path)

NAME_FIELD = 'ADM3_EN' if 'ADM3_EN' in gdf.columns else gdf.columns[0]
print('Naming files using field:', NAME_FIELD)


## 4. Download Dynamic World composites
One GeoTIFF per district, per year. Years before mid-2015 (no Dynamic World coverage) are skipped automatically and reported.

In [ ]:
YEARS = [2014, 2019, 2024]
DW_OUT_DIR = '/content/DynamicWorld_Output'

downloaded_files, skipped = download_dynamicworld_for_years(
    gdf=gdf, name_field=NAME_FIELD, years=YEARS, out_dir=DW_OUT_DIR,
)


## 5. Generate styled PNG maps

In [ ]:
MAPS_OUT_DIR = '/content/DynamicWorld_Maps_PNG'

generated_maps = generate_all_maps(
    in_dir=DW_OUT_DIR, out_dir=MAPS_OUT_DIR, class_dict=DW_CLASSES,
)


## 6. Compute per-class pixel/area statistics

In [ ]:
STATS_OUT_DIR = '/content/DynamicWorld_Stats'
stats_csv = os.path.join(STATS_OUT_DIR, 'DynamicWorld_Statistics.csv')

stats_df = build_statistics_table(
    in_dir=DW_OUT_DIR, class_names=DW_CLASS_NAMES, out_csv_path=stats_csv,
)
stats_df.head(10)


## 7. Build a wide pivot table (optional)

In [ ]:
pivot_csv = os.path.join(STATS_OUT_DIR, 'DynamicWorld_Statistics_Pivot_km2.csv')
pivot_df = build_pivot_table(stats_df, out_csv_path=pivot_csv, value_col='area_km2')
pivot_df.head(10)


## 8. Generate comparison charts

In [ ]:
CHARTS_OUT_DIR = '/content/DynamicWorld_Charts'
generated_charts = generate_all_charts(stats_df, out_dir=CHARTS_OUT_DIR)


## 9. Zip everything and download it to your computer

In [ ]:
zip_and_download(
    [stats_csv, pivot_csv] + generated_maps + generated_charts,
    zip_path='/content/DynamicWorld_All_Outputs.zip',
)
